# Agentic Stock Forecasting System - Debug Workflow

Notebook này được dựng từ `main.py` và `notebooks/blueprint.ipynb` để bạn debug từng phần của pipeline:

1. Live data ingestion từ `vnstock`
2. Processing, feature engineering và lưu SQLite
3. LightGBM Quantile forecasting và holdout metrics
4. LangGraph agent evaluation theo từng node hoặc full graph
5. Reporting ra JSON, Markdown, HTML

Các cell có tác dụng phụ như gọi API, sửa config hoặc sinh report đều có cờ bật/tắt riêng.

## 0. Pipeline Map

`main.py` đang chạy luồng sau:

`get_vingroup_and_context_data` -> `process_and_save_data` -> `load_from_sqlite(processed_<ticker>)` -> `generate_7_day_forecast` -> `build_agent_graph().invoke` -> `generate_reports`

In [1]:
import os
import sys
import json
import sqlite3
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import yaml
from dotenv import load_dotenv
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv()
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

print("Project root:", PROJECT_ROOT)
print("Google API key loaded:", bool(os.getenv("GOOGLE_API_KEY")))

Project root: c:\Users\Minh Man\Desktop\repos-vsc\pylab\pylab - Project\AGENTIC_STOCK_FORECASTING_SYSTEM
Google API key loaded: True


In [2]:
# Chỉnh các biến này khi debug.
TARGET_TICKER = "VPL"  # VIC, VHM, VRE, VPL
LOOKBACK_DAYS = 2 * 365

# True: gọi vnstock để lấy dữ liệu mới. False: đọc raw_* có sẵn từ data/database.sqlite.
USE_LIVE_DATA = True

# True: chạy lại processing và ghi đè các bảng raw_*/processed_* trong SQLite.
RUN_PROCESSING = True

# True: gọi LLM/Gemini trong agent. Nên để False khi chỉ debug data/model.
RUN_AGENT_LLM = True

# True: chạy full LangGraph. Có thể gọi LLM và news RSS tùy nhánh.
RUN_FULL_AGENT_GRAPH = True

# True: sinh report mới trong reports/json, reports/markdown, reports/html.
RUN_REPORTING = True

end_date = datetime.now().strftime("%Y-%m-%d")
start_date = (datetime.now() - timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%d")
start_date, end_date

('2024-05-12', '2026-05-12')

In [3]:
from src.ingestion.vnstock_api import get_vingroup_and_context_data
from src.processing.cleaner import process_and_save_data
from src.processing.features import generate_technical_features, generate_context_features
from src.processing.db_manager import load_from_sqlite, DB_PATH
from src.modeling.trainer import QuantileLightGBM
from src.modeling.predictor import generate_7_day_forecast
from src.reporting.generator import generate_reports

print("Project modules imported successfully.")

**Vui lòng chuyển đổi sang Vnstock3** thế hệ mới (3.2.1) với câu lệnh: `pip install vnstock3 --upgrade`.
**Từ 1/1/2025, vnstock3 sẽ được cài đặt khi sử dụng cú pháp** `pip install vnstock` **thay cho Vnstock Legacy** hiện tại.
Xem chi tiết [chuyển đổi sang vnstock3](https://vnstocks.com/docs/tai-lieu/migration-chuyen-doi-sang-vnstock3).
Phiên bản **Vnstock Legacy (0.2.9.2.3)** bạn đang sử dụng **sẽ không được nâng cấp thêm.**
Từ 7/10/2024 Vnstock giới thiệu nhóm Facebook Cộng đồng Vnstock, tham gia thảo luận tại đây: https://www.facebook.com/groups/vnstock.official

Project modules imported successfully.


In [4]:
def list_sqlite_tables(db_path=DB_PATH):
    path = Path(db_path)
    if not path.exists():
        return []
    with sqlite3.connect(path) as con:
        rows = con.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()
    return [row[0] for row in rows]


def load_sqlite_table(table_name, db_path=DB_PATH):
    with sqlite3.connect(db_path) as con:
        df = pd.read_sql_query(f"SELECT * FROM {table_name}", con)
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
        df = df.set_index("date").sort_index()
    return df


def summarize_frames(data_dict):
    rows = []
    for key, df in data_dict.items():
        rows.append({
            "symbol": key,
            "rows": len(df),
            "cols": len(df.columns),
            "start": df.index.min() if len(df) else None,
            "end": df.index.max() if len(df) else None,
            "missing_cells": int(df.isna().sum().sum()) if len(df) else 0,
        })
    return pd.DataFrame(rows)


def forecast_to_frame(forecast_result):
    df = pd.DataFrame(forecast_result["forecasts"])
    ordered_cols = ["step", "q_0.025", "q_0.1", "q_0.5", "q_0.9", "q_0.975"]
    return df[ordered_cols]


def display_metrics(metrics):
    display(pd.DataFrame([metrics]).T.rename(columns={0: "value"}))


print("SQLite tables:", list_sqlite_tables())

SQLite tables: ['processed_VHM', 'processed_VIC', 'processed_VPL', 'processed_VRE', 'raw_VHM', 'raw_VIC', 'raw_VN30', 'raw_VN30F1M', 'raw_VPL', 'raw_VRE']


## 1. Inspect Configs

Cell này đọc cấu hình model và agent. Lưu ý: `tool_adjust_model_hyperparams` trong agent có thể sửa `configs/model_config.yaml`, nên các cell retrain agent được để riêng.

In [5]:
with open("configs/model_config.yaml", "r", encoding="utf-8") as f:
    model_config = yaml.safe_load(f)

with open("configs/agent_config.yaml", "r", encoding="utf-8") as f:
    agent_config = yaml.safe_load(f)

display(Markdown("### model_config.yaml"))
display(model_config)
display(Markdown("### agent_config.yaml"))
display(agent_config)

### model_config.yaml

{'lightgbm_params': {'boosting_type': 'gbdt',
  'learning_rate': 0.15,
  'max_depth': 6,
  'n_estimators': 50,
  'num_leaves': 15,
  'random_state': 42,
  'verbose': -1}}

### agent_config.yaml

{'thresholds': {'max_mape': 0.03, 'max_retries': 2},
 'prompts': {'evaluate': 'Bạn là chuyên gia định lượng. Đánh giá kết quả dự báo cổ phiếu {ticker}.',
  'contextualize': 'Tìm tin tức giải thích cho biến động bất thường của mã {ticker}.'}}

## 2. Data Ingestion

Nếu `USE_LIVE_DATA=True`, cell này gọi `vnstock`. Nếu muốn debug offline, đặt `USE_LIVE_DATA=False` để đọc các bảng `raw_*` từ SQLite.

In [6]:
symbols = ["VIC", "VHM", "VRE", "VPL", "VN30", "VN30F1M"]

if USE_LIVE_DATA:
    raw_data = get_vingroup_and_context_data(start_date, end_date)
else:
    raw_data = {}
    for sym in symbols:
        table = f"raw_{sym}"
        if table in list_sqlite_tables():
            raw_data[sym] = load_sqlite_table(table)

print("Loaded symbols:", sorted(raw_data.keys()))
display(summarize_frames(raw_data))

2026-05-12 20:31:17 - DataIngestion - INFO - === BẮT ĐẦU QUÁ TRÌNH KÉO DỮ LIỆU TỪ VNSTOCK ===
2026-05-12 20:31:17 - DataIngestion - INFO - Đang kéo dữ liệu cho mã: VIC (Loại: stock) từ 2024-05-12 đến 2026-05-12
2026-05-12 20:31:17 - DataIngestion - INFO - ✅ Thành công! Đã lấy 498 dòng dữ liệu cho VIC.
2026-05-12 20:31:17 - DataIngestion - INFO - Đang kéo dữ liệu cho mã: VHM (Loại: stock) từ 2024-05-12 đến 2026-05-12
2026-05-12 20:31:18 - DataIngestion - INFO - ✅ Thành công! Đã lấy 498 dòng dữ liệu cho VHM.
2026-05-12 20:31:18 - DataIngestion - INFO - Đang kéo dữ liệu cho mã: VRE (Loại: stock) từ 2024-05-12 đến 2026-05-12
2026-05-12 20:31:18 - DataIngestion - INFO - ✅ Thành công! Đã lấy 498 dòng dữ liệu cho VRE.
2026-05-12 20:31:18 - DataIngestion - INFO - Đang kéo dữ liệu cho mã: VPL (Loại: stock) từ 2024-05-12 đến 2026-05-12
2026-05-12 20:31:18 - DataIngestion - INFO - ✅ Thành công! Đã lấy 249 dòng dữ liệu cho VPL.
2026-05-12 20:31:18 - DataIngestion - INFO - Đang kéo dữ liệu cho mã: 

Loaded symbols: ['VHM', 'VIC', 'VN30', 'VN30F1M', 'VPL', 'VRE']


,symbol,rows,cols,start,end,missing_cells
0,VIC,498,6,2024-05-13,2026-05-12,0
1,VHM,498,6,2024-05-13,2026-05-12,0
2,VRE,498,6,2024-05-13,2026-05-12,0
3,VPL,249,6,2025-05-13,2026-05-12,0
4,VN30,497,6,2024-05-13,2026-05-12,0
5,VN30F1M,498,6,2024-05-13,2026-05-12,0


In [7]:
assert TARGET_TICKER in raw_data, f"Missing {TARGET_TICKER}. Check USE_LIVE_DATA or raw_{TARGET_TICKER} table."

df_raw_target = raw_data[TARGET_TICKER].copy()
display(df_raw_target.head())
display(df_raw_target.tail())
display(df_raw_target.describe(include="all"))
display(df_raw_target.isna().sum().rename("missing"))

,open,high,low,close,volume,ticker
date,,,,,,
2025-05-13,85500,85500,85500,85500,4800,VPL
2025-05-14,91400,91400,91400,91400,2100100,VPL
2025-05-15,97700,97700,97700,97700,1815600,VPL
2025-05-16,104500,104500,93300,101000,1163300,VPL
2025-05-19,100000,101000,96400,98200,381300,VPL


,open,high,low,close,volume,ticker
date,,,,,,
2026-05-06,89000,92000,89000,90000,1322600,VPL
2026-05-07,91000,92000,88000,91500,1519200,VPL
2026-05-08,92000,92000,89000,91500,995600,VPL
2026-05-11,91500,93000,89000,89000,1121800,VPL
2026-05-12,88000,89800,86300,89500,718900,VPL


,open,high,low,close,volume,ticker
count,249.000000,249.000000,249.000000,249.000000,2.490000e+02,249
unique,NaN,NaN,NaN,NaN,NaN,1
top,NaN,NaN,NaN,NaN,NaN,VPL
freq,NaN,NaN,NaN,NaN,NaN,249
mean,85942.168675,87415.261044,84298.393574,85922.088353,6.691141e+05,NaN
std,7804.771914,8065.168540,7294.291669,7487.836511,6.142765e+05,NaN
min,70700.000000,71700.000000,67000.000000,70500.000000,4.800000e+03,NaN
25%,80200.000000,80900.000000,79600.000000,80300.000000,3.173000e+05,NaN
50%,85000.000000,86600.000000,83500.000000,85000.000000,4.732000e+05,NaN
75%,91500.000000,93000.000000,89200.000000,91700.000000,8.557000e+05,NaN


open      0
high      0
low       0
close     0
volume    0
ticker    0
Name: missing, dtype: int64

## 3. Processing & Feature Engineering

`process_and_save_data(raw_data)` tạo context features cho VN30/VN30F1M, tạo technical features cho từng mã cổ phiếu, merge context, drop NaN và lưu `processed_<ticker>` vào SQLite.

In [8]:
# Debug feature engineering trên 1 mã trước khi ghi DB.
df_feature_preview = generate_technical_features(df_raw_target)

if "VN30" in raw_data and "VN30F1M" in raw_data:
    vn30_features = generate_context_features(raw_data["VN30"], prefix="vn30")
    vn30f_features = generate_context_features(raw_data["VN30F1M"], prefix="vn30f")
    context_combined = vn30_features.join(vn30f_features, how="outer")
    df_feature_preview = df_feature_preview.join(context_combined, how="left")

display(df_feature_preview.head(20))
display(df_feature_preview.isna().sum().sort_values(ascending=False).head(20))

,open,high,low,close,volume,ticker,daily_return,vol_change,ma_7,ma_14,volatility_7,rsi_14,macd,atr_14,roc_7,day_of_week,month,close_lag_1,return_lag_1,close_lag_3,return_lag_3,close_lag_7,return_lag_7,close_lag_14,return_lag_14,vn30_return,vn30_close,vn30f_return,vn30f_close
date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2025-05-13,85500,85500,85500,85500,4800,VPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,1,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.007828,1382.78,0.004367,1380.0
2025-05-14,91400,91400,91400,91400,2100100,VPL,0.069006,436.520833,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,2,5,85500.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.010913,1397.87,0.011957,1396.5
2025-05-15,97700,97700,97700,97700,1815600,VPL,0.068928,-0.135470,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,3,5,91400.0,0.069006,NaN,NaN,NaN,NaN,NaN,NaN,0.002590,1401.49,0.003008,1400.7
2025-05-16,104500,104500,93300,101000,1163300,VPL,0.033777,-0.359275,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,4,5,97700.0,0.068928,85500.0,NaN,NaN,NaN,NaN,NaN,-0.012166,1384.44,-0.011780,1384.2
2025-05-19,100000,101000,96400,98200,381300,VPL,-0.027723,-0.672226,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,0,5,101000.0,0.033777,91400.0,0.069006,NaN,NaN,NaN,NaN,-0.003388,1379.75,-0.002962,1380.1
2025-05-20,95100,102000,95100,99500,617200,VPL,0.013238,0.618673,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,1,5,98200.0,-0.027723,97700.0,0.068928,NaN,NaN,NaN,NaN,0.020127,1407.52,0.020216,1408.0
2025-05-21,99700,101300,95700,98000,396300,VPL,-0.015075,-0.357907,95900.000000,NaN,NaN,NaN,NaN,0.000000,NaN,2,5,99500.0,0.013238,101000.0,0.033777,NaN,NaN,NaN,NaN,0.008412,1419.36,0.004616,1414.5
2025-05-22,96500,96500,92000,92000,597200,VPL,-0.061224,0.506939,96828.571429,NaN,0.049392,NaN,NaN,0.000000,7.602339,3,5,98000.0,-0.015075,98200.0,-0.027723,85500.0,NaN,NaN,NaN,-0.006905,1409.56,-0.008837,1402.0
2025-05-23,90500,92600,90300,90300,196400,VPL,-0.018478,-0.671132,96671.428571,NaN,0.043102,NaN,NaN,0.000000,-1.203501,4,5,92000.0,-0.061224,99500.0,0.013238,91400.0,0.069006,NaN,NaN,-0.000114,1409.40,0.003210,1406.5


macd             25
return_lag_14    15
close_lag_14     14
ma_14            13
rsi_14           13
return_lag_7      8
volatility_7      7
close_lag_7       7
roc_7             7
ma_7              6
return_lag_3      4
close_lag_3       3
return_lag_1      2
vol_change        1
daily_return      1
close_lag_1       1
vn30_return       0
vn30_close        0
vn30f_return      0
open              0
dtype: int64

In [9]:
if RUN_PROCESSING:
    processed_data = process_and_save_data(raw_data)
else:
    processed_data = {}
    for sym in [s for s in symbols if s not in ["VN30", "VN30F1M"]]:
        table = f"processed_{sym}"
        if table in list_sqlite_tables():
            processed_data[sym] = load_sqlite_table(table)

display(summarize_frames(processed_data))
print("SQLite tables after processing:", list_sqlite_tables())

2026-05-12 20:33:22 - DataProcessing - INFO - === BẮT ĐẦU QUÁ TRÌNH XỬ LÝ DỮ LIỆU (FEATURE ENGINEERING) ===
2026-05-12 20:33:22 - DBManager - INFO - 💾 Đã lưu thành công bảng 'raw_VN30' vào DB. (Số dòng: 497)
2026-05-12 20:33:23 - DBManager - INFO - 💾 Đã lưu thành công bảng 'raw_VN30F1M' vào DB. (Số dòng: 498)
2026-05-12 20:33:23 - DBManager - INFO - 💾 Đã lưu thành công bảng 'raw_VIC' vào DB. (Số dòng: 498)
2026-05-12 20:33:23 - DataProcessing - INFO - [VIC] Đã tạo Features & Merge Context. Xóa 26 dòng bị NaN. Còn lại: 472 dòng.
2026-05-12 20:33:23 - DBManager - INFO - 💾 Đã lưu thành công bảng 'processed_VIC' vào DB. (Số dòng: 472)
2026-05-12 20:33:23 - DBManager - INFO - 💾 Đã lưu thành công bảng 'raw_VHM' vào DB. (Số dòng: 498)
2026-05-12 20:33:23 - DataProcessing - INFO - [VHM] Đã tạo Features & Merge Context. Xóa 26 dòng bị NaN. Còn lại: 472 dòng.
2026-05-12 20:33:23 - DBManager - INFO - 💾 Đã lưu thành công bảng 'processed_VHM' vào DB. (Số dòng: 472)
2026-05-12 20:33:23 - DBManager -

,symbol,rows,cols,start,end,missing_cells
0,VIC,472,29,2024-06-17,2026-05-12,0
1,VHM,472,29,2024-06-17,2026-05-12,0
2,VRE,472,29,2024-06-17,2026-05-12,0
3,VPL,224,29,2025-06-17,2026-05-12,0


SQLite tables after processing: ['processed_VHM', 'processed_VIC', 'processed_VPL', 'processed_VRE', 'raw_VHM', 'raw_VIC', 'raw_VN30', 'raw_VN30F1M', 'raw_VPL', 'raw_VRE']


In [10]:
df_processed = load_from_sqlite(f"processed_{TARGET_TICKER}")
assert not df_processed.empty, f"processed_{TARGET_TICKER} is empty. Run processing first."

display(df_processed.head())
display(df_processed.tail())
print("Shape:", df_processed.shape)
print("Columns:", df_processed.columns.tolist())
display(df_processed.isna().sum().sort_values(ascending=False).head(20))

,open,high,low,close,volume,ticker,daily_return,vol_change,ma_7,ma_14,volatility_7,rsi_14,macd,atr_14,roc_7,day_of_week,month,close_lag_1,return_lag_1,close_lag_3,return_lag_3,close_lag_7,return_lag_7,close_lag_14,return_lag_14,vn30_return,vn30_close,vn30f_return,vn30f_close
date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2025-06-17,94100,96000,93900,95400,125700,VPL,0.014894,-0.562935,90057.142857,89092.857143,0.024552,69.915990,646.607852,3024.540644,8.163265,1,6,94000.0,0.065760,88200.0,0.000000,88200.0,0.0,88000.0,0.009174,0.007773,1431.39,0.003035,1421.0
2025-06-18,95900,95900,94200,94300,92300,VPL,-0.011530,-0.265712,90928.571429,89507.142857,0.025809,64.985479,916.912329,2929.930598,6.916100,2,6,95400.0,0.014894,88200.0,0.000000,88200.0,0.0,88500.0,0.005682,0.001118,1432.99,0.004504,1427.4
2025-06-19,90000,95300,90000,94200,47500,VPL,-0.001060,-0.485374,91785.714286,89878.571429,0.025879,64.539889,1110.262989,3099.221270,6.802721,3,6,94300.0,-0.011530,94000.0,0.065760,88200.0,0.0,89000.0,0.005650,0.004403,1439.30,0.009388,1440.8
2025-06-20,93100,94200,93100,93500,73900,VPL,-0.007431,0.555789,92542.857143,90307.142857,0.026490,61.367819,1193.255523,2956.419750,6.009070,4,6,94200.0,-0.001060,95400.0,0.014894,88200.0,0.0,87500.0,-0.016854,-0.002856,1435.19,-0.011521,1424.2
2025-06-23,93300,94500,91300,94500,34700,VPL,0.010695,-0.530447,93442.857143,90807.142857,0.026214,64.083596,1324.451890,2973.818340,7.142857,0,6,93500.0,-0.007431,94300.0,-0.011530,88200.0,0.0,87500.0,0.000000,0.009149,1448.32,0.006600,1433.6


,open,high,low,close,volume,ticker,daily_return,vol_change,ma_7,ma_14,volatility_7,rsi_14,macd,atr_14,roc_7,day_of_week,month,close_lag_1,return_lag_1,close_lag_3,return_lag_3,close_lag_7,return_lag_7,close_lag_14,return_lag_14,vn30_return,vn30_close,vn30f_return,vn30f_close
date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-05-06,89000,92000,89000,90000,1322600,VPL,0.022727,-0.350616,85457.142857,84571.428571,0.022146,63.791776,1420.156847,4019.693289,7.270560,2,5,88000.0,0.032864,85700.0,0.016607,83900.0,0.002389,80200.0,-0.009877,0.011338,2053.41,0.010398,2060.0
2026-05-07,91000,92000,88000,91500,1519200,VPL,0.016667,0.148647,86600.000000,85392.857143,0.021178,66.072221,1808.308049,4018.286626,9.580838,3,5,90000.0,0.022727,85200.0,-0.005834,83500.0,-0.004768,80000.0,-0.002494,0.012511,2079.10,0.009709,2080.0
2026-05-08,92000,92000,89000,91500,995600,VPL,0.000000,-0.344655,88028.571429,85928.571429,0.015253,66.072221,2091.807616,3945.551867,12.269939,4,5,91500.0,0.016667,88000.0,0.032864,81500.0,-0.023952,84000.0,0.050000,-0.002424,2074.06,-0.003173,2073.4
2026-05-11,91500,93000,89000,89000,1121800,VPL,-0.027322,0.126758,88700.000000,86135.714286,0.020361,58.901586,2090.654199,3949.441019,5.575326,0,5,91500.0,0.000000,90000.0,0.022727,84300.0,0.034356,86100.0,0.025000,-0.016176,2040.51,-0.016591,2039.0
2026-05-12,88000,89800,86300,89500,718900,VPL,0.005618,-0.359155,89242.857143,86614.285714,0.020003,59.840322,2105.811446,3917.338089,4.434072,1,5,89000.0,-0.027322,91500.0,0.016667,85700.0,0.016607,82800.0,-0.038328,0.006596,2053.97,0.004904,2049.0


Shape: (224, 29)
Columns: ['open', 'high', 'low', 'close', 'volume', 'ticker', 'daily_return', 'vol_change', 'ma_7', 'ma_14', 'volatility_7', 'rsi_14', 'macd', 'atr_14', 'roc_7', 'day_of_week', 'month', 'close_lag_1', 'return_lag_1', 'close_lag_3', 'return_lag_3', 'close_lag_7', 'return_lag_7', 'close_lag_14', 'return_lag_14', 'vn30_return', 'vn30_close', 'vn30f_return', 'vn30f_close']


open             0
day_of_week      0
vn30f_return     0
vn30_close       0
vn30_return      0
return_lag_14    0
close_lag_14     0
return_lag_7     0
close_lag_7      0
return_lag_3     0
close_lag_3      0
return_lag_1     0
close_lag_1      0
month            0
roc_7            0
high             0
atr_14           0
macd             0
rsi_14           0
volatility_7     0
dtype: int64

## 4. Modeling Debug

Model dùng LightGBM quantile regression. Target training là future return: `(close[t+step] - close[t]) / close[t]`, sau đó khôi phục lại giá dự báo.

In [11]:
trainer = QuantileLightGBM()
X_step1, y_step1 = trainer.prepare_data(df_processed, step=1)

print("Feature matrix:", X_step1.shape)
print("Target:", y_step1.shape)
print("Feature columns:", trainer.features)
display(X_step1.tail())
display(y_step1.tail().rename("target_return_t_plus_1"))

Feature matrix: (223, 28)
Target: (223,)
Feature columns: ['open', 'high', 'low', 'close', 'volume', 'daily_return', 'vol_change', 'ma_7', 'ma_14', 'volatility_7', 'rsi_14', 'macd', 'atr_14', 'roc_7', 'day_of_week', 'month', 'close_lag_1', 'return_lag_1', 'close_lag_3', 'return_lag_3', 'close_lag_7', 'return_lag_7', 'close_lag_14', 'return_lag_14', 'vn30_return', 'vn30_close', 'vn30f_return', 'vn30f_close']


,open,high,low,close,volume,daily_return,vol_change,ma_7,ma_14,volatility_7,rsi_14,macd,atr_14,roc_7,day_of_week,month,close_lag_1,return_lag_1,close_lag_3,return_lag_3,close_lag_7,return_lag_7,close_lag_14,return_lag_14,vn30_return,vn30_close,vn30f_return,vn30f_close
date,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-05-05,84100,90300,84000,88000,2036700,0.032864,2.508527,84585.714286,83871.428571,0.021568,60.505079,1056.805213,4021.208158,5.137395,1,5,85200.0,-0.005834,84300.0,0.034356,83700.0,-0.018757,81000.0,0.012500,0.008433,2030.39,0.010357,2038.8
2026-05-06,89000,92000,89000,90000,1322600,0.022727,-0.350616,85457.142857,84571.428571,0.022146,63.791776,1420.156847,4019.693289,7.270560,2,5,88000.0,0.032864,85700.0,0.016607,83900.0,0.002389,80200.0,-0.009877,0.011338,2053.41,0.010398,2060.0
2026-05-07,91000,92000,88000,91500,1519200,0.016667,0.148647,86600.000000,85392.857143,0.021178,66.072221,1808.308049,4018.286626,9.580838,3,5,90000.0,0.022727,85200.0,-0.005834,83500.0,-0.004768,80000.0,-0.002494,0.012511,2079.10,0.009709,2080.0
2026-05-08,92000,92000,89000,91500,995600,0.000000,-0.344655,88028.571429,85928.571429,0.015253,66.072221,2091.807616,3945.551867,12.269939,4,5,91500.0,0.016667,88000.0,0.032864,81500.0,-0.023952,84000.0,0.050000,-0.002424,2074.06,-0.003173,2073.4
2026-05-11,91500,93000,89000,89000,1121800,-0.027322,0.126758,88700.000000,86135.714286,0.020361,58.901586,2090.654199,3949.441019,5.575326,0,5,91500.0,0.000000,90000.0,0.022727,84300.0,0.034356,86100.0,0.025000,-0.016176,2040.51,-0.016591,2039.0


date
2026-05-05    0.022727
2026-05-06    0.016667
2026-05-07    0.000000
2026-05-08   -0.027322
2026-05-11    0.005618
Name: target_return_t_plus_1, dtype: float64

In [12]:
metrics = trainer.evaluate_holdout(df_processed)
display_metrics(metrics)

2026-05-12 20:35:08 - ModelTrainer - INFO - [VPL] Đang đánh giá Holdout (30 ngày)...
2026-05-12 20:35:08 - ModelTrainer - INFO - [VPL] Holdout Metrics -> MAE: 1596.85 | RMSE: 2217.26 | MAPE: 0.0190


,value
MAE,1596.853015
RMSE,2217.263330
MAPE,0.019003


In [13]:
forecast_result = generate_7_day_forecast(df_processed)
forecast_df = forecast_to_frame(forecast_result)

display_metrics(forecast_result["metrics"])
display(forecast_df)

2026-05-12 20:35:24 - Predictor - INFO - === BẮT ĐẦU DỰ BÁO 7 NGÀY CHO VPL ===
2026-05-12 20:35:24 - ModelTrainer - INFO - [VPL] Đang đánh giá Holdout (30 ngày)...
2026-05-12 20:35:24 - ModelTrainer - INFO - [VPL] Holdout Metrics -> MAE: 1596.85 | RMSE: 2217.26 | MAPE: 0.0190
2026-05-12 20:35:24 - Predictor - INFO - [VPL] Đang chạy Model cho 7 steps (5 quantiles/step)...
2026-05-12 20:35:24 - Predictor - INFO - === HOÀN TẤT DỰ BÁO CHO VPL ===


,value
MAE,1596.853015
RMSE,2217.263330
MAPE,0.019003


,step,q_0.025,q_0.1,q_0.5,q_0.9,q_0.975
0,1,86575.610431,87464.843792,89603.453896,91115.423556,91226.217857
1,2,82634.680562,84682.962685,87572.224432,90915.260593,92408.443841
2,3,80892.190873,84042.881627,89468.174326,91269.824033,93385.710768
3,4,81539.451389,81154.066789,86560.777479,89739.298397,94987.036380
4,5,82247.304373,83051.110106,87692.089910,89489.245018,92527.162334
5,6,80959.346637,81681.244782,88776.740635,93056.220201,91492.235258
6,7,77535.034354,82565.552222,87555.300145,92688.249274,91801.573186


In [14]:
import plotly.graph_objects as go

steps = [f"T+{int(s)}" for s in forecast_df["step"]]
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=steps + steps[::-1],
    y=forecast_df["q_0.975"].tolist() + forecast_df["q_0.025"].tolist()[::-1],
    fill="toself",
    fillcolor="rgba(99, 110, 250, 0.18)",
    line=dict(color="rgba(255,255,255,0)"),
    name="95% interval",
))
fig.add_trace(go.Scatter(
    x=steps + steps[::-1],
    y=forecast_df["q_0.9"].tolist() + forecast_df["q_0.1"].tolist()[::-1],
    fill="toself",
    fillcolor="rgba(0, 204, 150, 0.28)",
    line=dict(color="rgba(255,255,255,0)"),
    name="80% interval",
))
fig.add_trace(go.Scatter(x=steps, y=forecast_df["q_0.5"], mode="lines+markers", name="median"))
fig.update_layout(title=f"{TARGET_TICKER} 7-day quantile forecast", xaxis_title="Forecast step", yaxis_title="Price")
fig.show()

## 5. Agent Evaluation Debug

Cell đầu tiên chỉ tạo `initial_state`. Các cell sau cho phép debug node an toàn trước. Những cell gọi Gemini hoặc có thể sửa config đều được đặt sau cờ `RUN_AGENT_LLM` hoặc `RUN_FULL_AGENT_GRAPH`.

In [15]:
initial_state = {
    "ticker": TARGET_TICKER,
    "forecast_data": forecast_result,
    "retry_count": 0,
}

display(initial_state.keys())
display_metrics(initial_state["forecast_data"]["metrics"])
display(forecast_to_frame(initial_state["forecast_data"]))

dict_keys(['ticker', 'forecast_data', 'retry_count'])

,value
MAE,1596.853015
RMSE,2217.263330
MAPE,0.019003


,step,q_0.025,q_0.1,q_0.5,q_0.9,q_0.975
0,1,86575.610431,87464.843792,89603.453896,91115.423556,91226.217857
1,2,82634.680562,84682.962685,87572.224432,90915.260593,92408.443841
2,3,80892.190873,84042.881627,89468.174326,91269.824033,93385.710768
3,4,81539.451389,81154.066789,86560.777479,89739.298397,94987.036380
4,5,82247.304373,83051.110106,87692.089910,89489.245018,92527.162334
5,6,80959.346637,81681.244782,88776.740635,93056.220201,91492.235258
6,7,77535.034354,82565.552222,87555.300145,92688.249274,91801.573186


In [16]:
# Chạy 2 node không cần LLM: validate quantile và evaluate metrics.
from src.agent.nodes import node_validate, node_evaluate
from src.agent.graph import route_after_evaluate

state_after_validate = node_validate(initial_state.copy())
state_after_evaluate = node_evaluate(state_after_validate.copy())

print("quantiles_fixed:", state_after_evaluate.get("quantiles_fixed"))
print("evaluation_status:", state_after_evaluate.get("evaluation_status"))
print("evaluation_reason:", state_after_evaluate.get("evaluation_reason"))
print("next route:", route_after_evaluate(state_after_evaluate))

final_state = state_after_evaluate.copy()

c:\Users\Minh Man\AppData\Local\Programs\Python\Python39\lib\site-packages\google\api_core\_python_version_support.py:242: FutureWarning:

You are using a non-supported Python version (3.9.13). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.

c:\Users\Minh Man\AppData\Local\Programs\Python\Python39\lib\site-packages\google\api_core\_python_version_support.py:242: FutureWarning:

You are using a non-supported Python version (3.9.13). Google will not post any further updates to google.ai.generativelanguage_v1beta supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.ai.generativelanguage_v1beta.

2026-05-12 20:36:24 - AgentNodes - INFO - === [NODE 1] KÍCH HOẠT: KIỂM TRA LUẬT THỊ TRƯỜNG & LỖI QUANTILE ===
2026-05-12 20:36:24 - AgentNodes - WARNING - ⚠️ Đã phát hiện và sửa ch

quantiles_fixed: True
evaluation_status: PASS
evaluation_reason: Mô hình ổn định. MAPE (1.90%) nằm trong mức cho phép.
next route: node_recommend


In [17]:
# Optional: gọi news RSS + Gemini. Bật RUN_AGENT_LLM=True khi bạn muốn debug node contextualize/recommend.
if RUN_AGENT_LLM:
    from src.agent.nodes import node_contextualize, node_improve, node_recommend
    from src.agent.graph import route_after_contextualize

    llm_state = state_after_evaluate.copy()
    if llm_state.get("evaluation_status") != "PASS":
        llm_state = node_contextualize(llm_state)
        print("shock_type:", llm_state.get("shock_type"))
        print("route after contextualize:", route_after_contextualize(llm_state))

        # Cẩn thận: node_improve có thể sửa configs/model_config.yaml qua tool_adjust_model_hyperparams.
        if route_after_contextualize(llm_state) == "node_improve":
            llm_state = node_improve(llm_state)

    llm_state = node_recommend(llm_state)
    final_state = llm_state
else:
    print("Skipped LLM agent nodes. Set RUN_AGENT_LLM=True to run them.")

2026-05-12 20:36:55 - AgentNodes - INFO - === [NODE 3] KÍCH HOẠT: TỔNG HỢP & RA QUYẾT ĐỊNH CUỐI CÙNG ===
2026-05-12 20:37:32 - AgentNodes - INFO - 💡 Khuyến nghị: Dưới đây là nhận xét dành cho nhà đầu tư:

"Đợt dự báo cho mã VPL đạt độ tin cậy cao với chỉ số sai số MAPE tối ưu (1.90%), cho thấy mô hình đang vận hành ổn định và phản ánh sát diễn biến thị trường. Nhà đầu tư có thể tự tin sử dụng kết quả này làm cơ sở tham chiếu cho chiến lược giao dịch ngắn hạn. Tuy nhiên, do thị trường luôn tiềm ẩn biến động bất ngờ, khuyến nghị quý nhà đầu tư vẫn nên kết hợp quản trị rủi ro chặt chẽ."


In [18]:
# Optional: chạy đúng LangGraph như main.py.
if RUN_FULL_AGENT_GRAPH:
    from src.agent.graph import build_agent_graph

    agent_app = build_agent_graph()
    final_state = agent_app.invoke(initial_state)
    display(final_state)
else:
    print("Skipped full LangGraph. Set RUN_FULL_AGENT_GRAPH=True to run it.")

2026-05-12 20:37:52 - AgentGraph - INFO - Khởi tạo sơ đồ Agentic Workflow...
2026-05-12 20:37:52 - AgentNodes - INFO - === [NODE 1] KÍCH HOẠT: KIỂM TRA LUẬT THỊ TRƯỜNG & LỖI QUANTILE ===
2026-05-12 20:37:52 - AgentNodes - INFO - === [NODE 2] KÍCH HOẠT: ĐÁNH GIÁ CHẤT LƯỢNG MÔ HÌNH ===
2026-05-12 20:37:52 - AgentNodes - INFO - ✅ PASS: Mô hình dự báo tốt.
2026-05-12 20:37:52 - AgentNodes - INFO - === [NODE 3] KÍCH HOẠT: TỔNG HỢP & RA QUYẾT ĐỊNH CUỐI CÙNG ===
2026-05-12 20:38:30 - AgentNodes - INFO - 💡 Khuyến nghị: Dưới đây là nhận xét dành cho nhà đầu tư:

"Đợt dự báo cho mã VPL đạt độ tin cậy cao với chỉ số sai số MAPE tối ưu (1.90%), cho thấy mô hình đang vận hành ổn định và phản ánh sát diễn biến thị trường. Nhà đầu tư có thể tự tin sử dụng kết quả này làm cơ sở tham chiếu cho chiến lược giao dịch ngắn hạn. Tuy nhiên, do thị trường luôn tiềm ẩn biến động bất ngờ, khuyến nghị quý nhà đầu tư vẫn nên duy trì quản trị rủi ro chặt chẽ."


{'ticker': 'VPL',
 'forecast_data': {'ticker': 'VPL',
  'metrics': {'MAE': 1596.8530154765617,
   'RMSE': 2217.2633295796168,
   'MAPE': 0.019002773493075015},
  'forecasts': [{'step': 1,
    'q_0.025': 86575.61043094815,
    'q_0.1': 87464.84379171058,
    'q_0.5': 89603.45389637633,
    'q_0.9': 91115.42355606265,
    'q_0.975': 91226.21785676929},
   {'step': 2,
    'q_0.025': 82634.68056199072,
    'q_0.1': 84682.96268535734,
    'q_0.5': 87572.22443234427,
    'q_0.9': 90915.26059305319,
    'q_0.975': 92408.44384106752},
   {'step': 3,
    'q_0.025': 80892.19087256327,
    'q_0.1': 84042.88162706528,
    'q_0.5': 89468.17432628576,
    'q_0.9': 91269.82403261971,
    'q_0.975': 93385.71076835944},
   {'step': 4,
    'q_0.025': 81154.06678869456,
    'q_0.1': 81539.45138873105,
    'q_0.5': 86560.77747948804,
    'q_0.9': 89739.29839683897,
    'q_0.975': 94987.03638048339},
   {'step': 5,
    'q_0.025': 82247.3043728018,
    'q_0.1': 83051.1101055723,
    'q_0.5': 87692.089909662

## 6. Reporting

`generate_reports(final_state)` xuất JSON, Markdown và HTML. Nếu không chạy LLM agent, cell dưới sẽ thêm recommendation placeholder để report vẫn sinh được.

In [19]:
report_state = final_state.copy()
report_state.setdefault("ticker", TARGET_TICKER)
report_state.setdefault("forecast_data", forecast_result)
report_state.setdefault("quantiles_fixed", False)
report_state.setdefault("evaluation_status", "NOT_RUN")
report_state.setdefault("evaluation_reason", "Agent recommendation was skipped in notebook debug mode.")
report_state.setdefault("news_context", "Skipped")
report_state.setdefault("shock_type", "NOT_RUN")
report_state.setdefault("news_analysis", "Skipped")
report_state.setdefault("action_taken", "Skipped")
report_state.setdefault("final_recommendation", "Notebook debug mode: review model metrics and forecast intervals manually.")

display(report_state.keys())
display(forecast_to_frame(report_state["forecast_data"]))

dict_keys(['ticker', 'forecast_data', 'quantiles_fixed', 'evaluation_status', 'evaluation_reason', 'retry_count', 'final_recommendation', 'news_context', 'shock_type', 'news_analysis', 'action_taken'])

,step,q_0.025,q_0.1,q_0.5,q_0.9,q_0.975
0,1,86575.610431,87464.843792,89603.453896,91115.423556,91226.217857
1,2,82634.680562,84682.962685,87572.224432,90915.260593,92408.443841
2,3,80892.190873,84042.881627,89468.174326,91269.824033,93385.710768
3,4,81154.066789,81539.451389,86560.777479,89739.298397,94987.036380
4,5,82247.304373,83051.110106,87692.089910,89489.245018,92527.162334
5,6,80959.346637,81681.244782,88776.740635,91492.235258,93056.220201
6,7,77535.034354,82565.552222,87555.300145,91801.573186,92688.249274


In [20]:
if RUN_REPORTING:
    generate_reports(report_state)
    print("Reports generated under reports/json, reports/markdown, reports/html")
else:
    print("Skipped reporting. Set RUN_REPORTING=True to generate files.")

2026-05-12 20:40:30 - ReportGenerator - INFO - 📄 Đã lưu báo cáo JSON: reports/json/VPL_report_2026-05-12.json
2026-05-12 20:40:30 - ReportGenerator - INFO - 📝 Đã lưu báo cáo Markdown: reports/markdown/VPL_report_2026-05-12.md
2026-05-12 20:40:31 - ReportGenerator - INFO - 🌐 Đã lưu báo cáo HTML: reports/html/VPL_report_2026-05-12.html


Reports generated under reports/json, reports/markdown, reports/html


## 7. One-Cell End-to-End Run

Cell này tương đương `main.py`, để cuối notebook vì nó chạy toàn bộ luồng và có thể gọi API/LLM/sinh report.

In [22]:
RUN_MAIN_EQUIVALENT = True

if RUN_MAIN_EQUIVALENT:
    from main import run_daily_pipeline
    run_daily_pipeline(target_ticker=TARGET_TICKER)
else:
    print("Skipped main.py equivalent run. Set RUN_MAIN_EQUIVALENT=True to execute the full daily pipeline.")

2026-05-12 20:44:24 - MainPipeline - INFO - =====================================================
2026-05-12 20:44:24 - MainPipeline - INFO - 🚀 BẮT ĐẦU CHẠY PIPELINE TỰ ĐỘNG CHO MÃ: VPL
2026-05-12 20:44:24 - MainPipeline - INFO - =====================================================
2026-05-12 20:44:24 - DataIngestion - INFO - === BẮT ĐẦU QUÁ TRÌNH KÉO DỮ LIỆU TỪ VNSTOCK ===
2026-05-12 20:44:24 - DataIngestion - INFO - Đang kéo dữ liệu cho mã: VIC (Loại: stock) từ 2024-05-12 đến 2026-05-12
2026-05-12 20:44:24 - DataIngestion - INFO - ✅ Thành công! Đã lấy 498 dòng dữ liệu cho VIC.
2026-05-12 20:44:24 - DataIngestion - INFO - Đang kéo dữ liệu cho mã: VHM (Loại: stock) từ 2024-05-12 đến 2026-05-12
2026-05-12 20:44:24 - DataIngestion - INFO - ✅ Thành công! Đã lấy 498 dòng dữ liệu cho VHM.
2026-05-12 20:44:24 - DataIngestion - INFO - Đang kéo dữ liệu cho mã: VRE (Loại: stock) từ 2024-05-12 đến 2026-05-12
2026-05-12 20:44:24 - DataIngestion - INFO - ✅ Thành công! Đã lấy 498 dòng dữ liệu cho 